# 🎬 ComfyUI + LTX-Video + Flux Cloud Backend para 'Historias y Momentazos'
Este notebook ejecuta **ComfyUI** con aceleración por GPU gratuita en Google Colab y crea un túnel público seguro (Cloudflare) para conectarlo directamente a tu panel web.

### Pasos para iniciar:
1. Ve a **Entorno de ejecución** > **Cambiar tipo de entorno de ejecución** y asegúrate de tener seleccionada **GPU T4**.
2. Presiona el botón de **Play (▶)** en la celda de abajo.
3. Espera ~2 minutos hasta que veas el enlace que termina en `.trycloudflare.com`.
4. Copia ese enlace y pégalo en los **Ajustes** de tu panel web en `ComfyUI Cloud URL`.

In [ ]:
# 1. Instalar ComfyUI y dependencias
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q -r requirements.txt

# 2. Instalar nodos de LTX-Video y Flux
!git clone https://github.com/kijai/ComfyUI-LTXVideo custom_nodes/ComfyUI-LTXVideo
!pip install -q -r custom_nodes/ComfyUI-LTXVideo/requirements.txt

# 3. Descargar modelo LTX-Video ultrarrápido (v0.9.1)
!mkdir -p models/checkpoints
!wget -q -c https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors -P models/checkpoints/

# 4. Descargar Cloudflared para crear el túnel público
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 5. Iniciar ComfyUI en segundo plano
import subprocess, time, re
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--highvram"])
time.sleep(5)

# 6. Iniciar túnel de Cloudflare y mostrar la URL pública
print("\n🚀 Iniciando túnel seguro de Cloudflare...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stderr=subprocess.PIPE, text=True)
for line in tunnel_proc.stderr:
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n=======================================================")
            print(f"✅ ¡TU URL DE COMFYUI CLOUD ESTÁ LISTA!")
            print(f"👉 {match.group(0)}")
            print("Copia esta URL y pégala en Ajustes de tu panel web.")
            print("=======================================================\n")
            break
